In [ ]:
import os, getpass


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")
_set_env("LANGCHAIN_API_KEY")
_set_env("MISTRAL_API_KEY")
_set_env("TOGETHER_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model


def _init_llm(llm_name: str, temperature: int = 0):
    if llm_name == "o3-mini-2025-01-31":
        return ChatOpenAI(model=llm_name)
    elif llm_name.startswith("gpt") or llm_name.startswith("openai"):
        return ChatOpenAI(model=llm_name, temperature=temperature)
    elif llm_name.lower().startswith(
        "meta-llama".lower()
    ) or llm_name.lower().startswith("mistralai".lower()):
        return init_chat_model(
            llm_name,
            model_provider="together",
            temperature=temperature,
        )
    elif llm_name.lower().startswith("mistral"):
        return init_chat_model(
            llm_name,
            model_provider="mistralai",
            temperature=temperature,
        )

In [12]:
import datetime
import pandas as pd

from tqdm import tqdm

mistral_model = "mistral-small-2503"  # "mistralai/Mixtral-8x7B-Instruct-v0.1",
llama_model =  "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# === Prompts ===
with open("rag/prompts/system_rag_new.txt", "r") as f:
    SYSTEM_PROMPT = f.read()

with open("rag/prompts/rag_new.txt", "r") as f:
    RAG_PROMPT_TEMPLATE = f.read()

with open("rag/prompts/zero-shot.txt", "r") as f:
    ZERO_SHOT_TEMPLATE = f.read()

with open("rag/prompts/redefine_rag.txt", "r") as f:
    REDEFINE_RAG = f.read()

# ===  models ===
# Llama 70B (example: OpenAI-compatible endpoint)
# Mistral small

def generate_w_structured_context():
    df = pd.read_csv("evaluation/samples/big_eval_new.csv")

    for c in ['rag_answer', 'zero_shot_answer', 'timestamp']:
        df[c] = ""

    i = 0
    all_answers = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        q_row = row.to_dict()
        q = row["question"]
        c = row["structured_context"]
        
        if row["model"].find("lama") > -1:
            llm = _init_llm(llm_name=llama_model, temperature=0.7)
            q_row["model"] = llama_model
        else:
            llm = _init_llm(llm_name=mistral_model, temperature=0.7)
            q_row["model"] = mistral_model
        

        q_row["rag_answer"]  = llm.invoke(
            [
                    SystemMessage(SYSTEM_PROMPT),
                    HumanMessage(RAG_PROMPT_TEMPLATE.format(question=q, context=c)),
                ],
        ).content

        q_row["zero_shot_answer"]  = llm.invoke(
            [
                    SystemMessage(SYSTEM_PROMPT),
                    HumanMessage(ZERO_SHOT_TEMPLATE.format(question=q)),
                ],
        ).content

        q_row["rag_two_steps_answer"]  = llm.invoke(
         [
                SystemMessage(SYSTEM_PROMPT),
                HumanMessage(REDEFINE_RAG.format(question=q, context=c, draft_answer=q_row["zero_shot_answer"])),
            ],
        ).content
        
        if i == 0:
            print(SYSTEM_PROMPT)
            print(ZERO_SHOT_TEMPLATE.format(question=q))
            print(RAG_PROMPT_TEMPLATE.format(question=q, context=c))
            i = i +1
        q_row["timestamp"]=datetime.datetime.now(tz=datetime.UTC).isoformat()

        all_answers.append(q_row)

    return pd.DataFrame(all_answers)

# Ran only once  
# generation_df = generate_w_structured_context()
# generation_df.to_csv("evaluation/samples/answers_w_structured_context.csv", index=False)
generation_df = pd.read_csv("evaluation/samples/answers_w_structured_context.csv")

In [14]:
generation_df.columns

Index(['idx', 'model', 'temperature', 'question', 'aql_params', 'query',
       'aql_results', 'context', 'rag_answer', 'zero_shot_answer', 'timestamp',
       'structured_context', 'rag_two_steps_answer'],
      dtype='object')

In [15]:
generation_df.head()

,idx,model,temperature,question,aql_params,query,aql_results,context,rag_answer,zero_shot_answer,timestamp,structured_context,rag_two_steps_answer
0,CLIMATE INDICATORS_Exploratory,mistral-small-2503,-1,"How do variations in cryospheric indicators, s...","{'n': 16, 'k': 3, 'k_threshold': 0.3, 's': 2}",cryospheric indicators AND glacier retreat AND...,[{'title': 'Rapid marine deglaciation: asynchr...,[Document(id='f2516234-7ed2-488a-b315-14b85e7a...,"Variations in cryospheric indicators, such as ...","Variations in cryospheric indicators, such as ...",2025-07-26T17:44:03.591002+00:00,\n# RELATED DOCUMENTS\n\n- Title: A Holistic A...,### Refined Answer\n\nVariations in cryospheri...
1,CLIMATE INDICATORS_Comparative,mistral-small-2503,-1,How do cryospheric indicators compare to land ...,"{'n': 16, 'k': 3, 'k_threshold': 0.3, 's': 2}",cryospheric indicators AND land surface indica...,[{'title': 'Surface Contribution to Planetary ...,[Document(id='df8584cd-0835-496c-98c1-22d326ab...,Cryospheric indicators and land surface/agricu...,Cryospheric indicators and land surface/agricu...,2025-07-26T17:44:12.971023+00:00,\n# RELATED DOCUMENTS\n\n- Title: Cryosphere a...,### Refined Answer\n\nCryospheric indicators a...
2,CLIMATE INDICATORS_Descriptive,mistral-small-2503,-1,What are cryospheric indicators and how do the...,"{'n': 16, 'k': 3, 'k_threshold': 0.3, 's': 2}",cryospheric indicators AND Earth's climate AND...,[{'title': 'Climates of the Earth and Cryosphe...,[Document(id='df8584cd-0835-496c-98c1-22d326ab...,### Answer\n\nCryospheric indicators are a set...,### Cryospheric Indicators and Their Influence...,2025-07-26T17:44:19.564318+00:00,\n# RELATED DOCUMENTS\n\n- Title: Cryosphere a...,### Cryospheric Indicators and Their Influence...
3,CLIMATE INDICATORS_Causal,mistral-small-2503,-1,How does the melting of glaciers and ice sheet...,"{'n': 16, 'k': 3, 'k_threshold': 0.3, 's': 2}",glacier melting AND ice sheet melting AND glob...,[{'title': 'Role of Snowfall Versus Air Temper...,[Document(id='593df856-6e7a-423c-a3f3-f3269761...,The melting of glaciers and ice sheets signifi...,The melting of glaciers and ice sheets signifi...,2025-07-26T17:44:27.040848+00:00,\n# RELATED DOCUMENTS\n\n- Title: Role of Snow...,### Refined Answer\n\nThe melting of glaciers ...
4,CLIMATE INDICATORS_Relational,mistral-small-2503,-1,What is the relationship between cryospheric c...,"{'n': 16, 'k': 3, 'k_threshold': 0.3, 's': 2}",cryospheric changes AND glacier melt AND snow ...,[{'title': 'Carbon dynamics shift in changing ...,[Document(id='f589344e-b28d-42dc-9cff-b606a58d...,"The relationship between cryospheric changes, ...","The relationship between cryospheric changes, ...",2025-07-26T17:44:42.502110+00:00,\n# RELATED DOCUMENTS\n\n- Title: The Disappea...,### Refined Answer\n\nThe relationship between...
